# **1. Analisis Exploratorio por tablas**

El primer paso, antes de cualquier analisis, es cargar las librerias y funciones que se usaran para el mismo analisis.

In [ ]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")

import sys

sys.path.append("../src")

from edafuncs import (
    analizar_tabla,
    heatmap_nulos,
    histogramas,
    boxplots,
    correlaciones
)

from edaload import cargar_csv, cargar_parquet

## 1.1  person.csv

### **1.1.1 Calidad de person.csv**

El archivo person.csv, se define de la siguiente manera:

- person_id: Identificador único anonimizado de cada paciente.
- gender_concept_id: Identificador estandarizado del sexo del paciente.
- year_of_birth: Año de nacimiento del paciente.
- race_concept_id: Identificador estandarizado asociado al grupo racial del paciente.
- ethnicity_concept_id: Identificador estandarizado asociado a la etnia del paciente.

In [ ]:
person = cargar_csv(
    "person.csv"
)

In [ ]:
analizar_tabla(person, "person")

In [ ]:
heatmap_nulos(person)

En este caso, para la tabla **person.csv**, no presenta ni datos nulos o duplicados, lo que significa que lo siguiente seria realizar el analisis estadistico para comprobar su distribucion.

In [ ]:
histogramas(person)

Por el momento, los histogramas representan de forma debida los datos. Gracias al diccionario de datos obtenido anteriormente, se refleja el comportamiento sin ninguna anomalia presente.

In [ ]:
boxplots(person)

Lo importante a resaltar es la presencia de datos atipicos en la variable **year_of_birth**, para estos datos, se considero eliminar los registros para evitar un sesgo, especialmente gracias a su naturaleza y valor extremo dentro de la variable en si.

In [ ]:
correlaciones(person)

### **1.1.2 Tratamiento de la variable **year_of_birth****


Se identificaron registros con valores iguales a 1 y años anteriores a 1900.

Estos valores generan edades clínicamente imposibles (>120 años), por lo que se consideran errores de calidad de datos o codificaciones especiales de valores desconocidos.

Para preservar los registros clínicos asociados y evitar pérdida de información, dichos valores serán tratados como faltantes (NaN) en las variables derivadas de edad.

In [ ]:
person.loc[
    person["year_of_birth"] <= 1900,
    ["person_id", "year_of_birth"]
].head(20)

In [ ]:
person.loc[
    person["year_of_birth"] < 1900
].shape

In [ ]:
person.loc[
    person["year_of_birth"] < 1900,
    "year_of_birth"
].value_counts().sort_index()

In [ ]:
person["year_of_birth_clean"] = (
    person["year_of_birth"]
)

person.loc[
    person["year_of_birth"] < 1900,
    "year_of_birth_clean"
] = pd.NA


In [ ]:
print("Registros inválidos:",
      (person["year_of_birth_clean"] < 1900).sum())

print("Nulos creados:",
      person["year_of_birth_clean"].isna().sum())

In [ ]:
person.drop("year_of_birth", axis=1, inplace=True)

In [ ]:
boxplots(person)

### **1.1.3 Exploracion y analisis de datos de person.csv**

Para esta seccion, se van a mappear aquellas variables que tienen la terminacion "concept_id", juntandolo con la tabla concept, la cual es un diccionario global para muchas variables en el dataset. Con esto, podemos entender los datos de una forma mas realista y con sentido.

In [ ]:
concept = cargar_csv(
    "concept.csv"
)

concept_dict = dict(
    zip(
        concept["concept_id"],
        concept["concept_name"]
    )
)

In [ ]:
person["gender"] = (
    person["gender_concept_id"]
    .map(concept_dict)
)

person["race"] = (
    person["race_concept_id"]
    .map(concept_dict)
)

person["ethnicity"] = (
    person["ethnicity_concept_id"]
    .map(concept_dict)
)

In [ ]:
person[
    [
        "gender_concept_id",
        "gender",
        "race_concept_id",
        "race",
        "ethnicity_concept_id",
        "ethnicity"
    ]
].head()

In [ ]:
person["gender"].isna().sum()

person["race"].isna().sum()

person["ethnicity"].isna().sum()

In [ ]:
from datetime import datetime

current_year = datetime.now().year

person["age"] = (
    current_year -
    person["year_of_birth_clean"]
)

plt.figure(figsize=(10,5))

sns.histplot(
    person["age"],
    bins=30,
    kde=True
)

plt.title("Distribución de Edad")
plt.xlabel("Edad")
plt.show()

In [ ]:
plt.figure(figsize=(8,4))

sns.countplot(
    data=person,
    x="gender"
)

plt.title("Distribución por Género")

plt.show()

In [ ]:
person["gender"].value_counts(normalize=True)

Se observa una distribución relativamente balanceada entre pacientes femeninos (53.1%) y masculinos (46.9%).

Solo el 0.01% de los registros no pudieron asociarse a un concepto válido dentro de la tabla de vocabulario OMOP, lo que indica una alta integridad referencial para la variable gender_concept_id.

In [ ]:
plt.figure(figsize=(10,5))

sns.boxplot(
    data=person,
    x="gender",
    y="age"
)

plt.title("Distribución de edad por género")

plt.show()

In [ ]:
plt.figure(figsize=(10,5))

sns.histplot(
    data=person,
    x="age",
    hue="gender",
    bins=30,
    kde=True
)

plt.title("Distribución de edad por género")

plt.show()

In [ ]:
top_race = (
    person["race"]
    .value_counts()
    .head(10)
)

plt.figure(figsize=(10,5))

sns.barplot(
    x=top_race.values,
    y=top_race.index
)

plt.title("Top 10 razas")

plt.show()

In [ ]:
person["race"].value_counts(normalize=True)

In [ ]:
top_eth = (
    person["ethnicity"]
    .value_counts()
    .head(10)
)

plt.figure(figsize=(10,5))

sns.barplot(
    x=top_eth.values,
    y=top_eth.index
)

plt.title("Top 10 etnias")

plt.show()

In [ ]:
person["ethnicity"].value_counts(normalize=True)

In [ ]:
tabla = pd.crosstab(
    person["gender"],
    person["ethnicity"]
)

plt.figure(figsize=(12,4))

sns.heatmap(
    tabla,
    annot=True,
    fmt=",.0f",
    cmap="Blues"
)

plt.title(
    "Género vs Etnia"
)

plt.show()

In [ ]:
tabla = pd.crosstab(
    person["gender"],
    person["race"]
)

plt.figure(figsize=(12,5))

sns.heatmap(
    tabla,
    annot=True,
    fmt=",.0f",
    cmap="Blues"
)

plt.title(
    "Género vs Raza"
)

plt.show()

In [ ]:
person["age_group"] = pd.cut(
    person["age"],
    bins=[0,18,30,40,50,60,70,80,90,120]
)

pivot = pd.crosstab(
    person["age_group"],
    person["gender"]
)

pivot.plot(
    kind="bar",
    stacked=True,
    figsize=(12,5)
)

plt.title(
    "Distribución etaria por género"
)

plt.show()

In [ ]:
# import pandas as pd
# import matplotlib.pyplot as plt

# # -----------------------------
# # Grupos etarios
# # -----------------------------

# person["age_group"] = pd.cut(
#     person["age"],
#     bins=[0,10,20,30,40,50,60,70,80,90,100,120],
#     right=False
# )

# # -----------------------------
# # Tabla para pirámide
# # -----------------------------

# pyramid = pd.crosstab(
#     person["age_group"],
#     person["gender"]
# )

# # Asegurar columnas
# for col in ["MALE", "FEMALE"]:
#     if col not in pyramid.columns:
#         pyramid[col] = 0

# # Totales
# pyramid["total"] = (
#     pyramid["MALE"] +
#     pyramid["FEMALE"]
# )

# total_persons = pyramid["total"].sum()

# pyramid["pct"] = (
#     pyramid["total"] /
#     total_persons * 100
# )

# # Hombres negativos
# male = -pyramid["MALE"]
# female = pyramid["FEMALE"]

# # -----------------------------
# # Colores más suaves
# # -----------------------------

# male_color = "#8C1414"
# female_color = "#3E952E"

# # -----------------------------
# # Gráfico
# # -----------------------------

# fig, ax = plt.subplots(
#     figsize=(12,8)
# )

# ax.barh(
#     pyramid.index.astype(str),
#     male,
#     color=male_color,
#     label="Male"
# )

# ax.barh(
#     pyramid.index.astype(str),
#     female,
#     color=female_color,
#     label="Female"
# )

# # Línea central

# ax.axvline(
#     0,
#     color="black",
#     linewidth=1
# )

# # -----------------------------
# # Etiquetas
# # -----------------------------

# for i, (
#     total,
#     pct
# ) in enumerate(
#     zip(
#         pyramid["total"],
#         pyramid["pct"]
#     )
# ):

#     ax.text(
#         0,
#         i,
#         f"{total:,}\n({pct:.1f}%)",
#         ha="center",
#         va="center",
#         fontsize=9,
#         fontweight="bold",
#         bbox=dict(
#             boxstyle="round,pad=0.3",
#             facecolor="white",
#             alpha=0.8
#         )
#     )

# # -----------------------------
# # Limpieza visual
# # -----------------------------

# ax.set_title(
#     "Pirámide poblacional de pacientes",
#     fontsize=16,
#     weight="bold"
# )

# ax.set_xlabel("")
# ax.set_ylabel("")

# ax.set_xticks([])

# ax.spines["top"].set_visible(False)
# ax.spines["right"].set_visible(False)
# ax.spines["bottom"].set_visible(True)

# ax.legend(
#     loc="lower right"
# )

# plt.tight_layout()

# # plt.show()

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

# =====================================
# Crear grupos etarios
# =====================================

person["age_group"] = pd.cut(
    person["age"],
    bins=[0,18,30,40,50,60,70,80,90,120],
    right=False
)

# =====================================
# Tabla para la pirámide
# =====================================

pyramid = pd.crosstab(
    person["age_group"],
    person["gender"]
)

# Asegurar columnas
for col in ["MALE", "FEMALE"]:
    if col not in pyramid.columns:
        pyramid[col] = 0

# Totales
pyramid["total"] = (
    pyramid["MALE"] +
    pyramid["FEMALE"]
)

total_population = pyramid["total"].sum()

# % sobre la población total
pyramid["pct_population"] = (
    pyramid["total"] /
    total_population * 100
)

# % dentro del grupo etario
pyramid["male_pct"] = (
    pyramid["MALE"] /
    pyramid["total"] * 100
)

pyramid["female_pct"] = (
    pyramid["FEMALE"] /
    pyramid["total"] * 100
)

# =====================================
# Valores para la gráfica
# =====================================

male = -pyramid["MALE"]
female = pyramid["FEMALE"]

# =====================================
# Colores
# =====================================

male_color = "#4472C4"
female_color = "#F6BD60"

# =====================================
# Plot
# =====================================

fig, ax = plt.subplots(
    figsize=(14, 8)
)

ax.barh(
    pyramid.index.astype(str),
    male,
    color=male_color,
    label="Male"
)

ax.barh(
    pyramid.index.astype(str),
    female,
    color=female_color,
    label="Female"
)

# Línea central

ax.axvline(
    0,
    color="black",
    linewidth=1
)

# =====================================
# Etiquetas
# =====================================

max_width = max(
    pyramid["MALE"].max(),
    pyramid["FEMALE"].max()
)

for i, row in enumerate(
    pyramid.itertuples()
):

    # Centro
    ax.text(
        0,
        i,
        f"{row.total:,}\n({row.pct_population:.1f}%)",
        ha="center",
        va="center",
        fontsize=9,
        fontweight="bold",
        bbox=dict(
            boxstyle="round,pad=0.3",
            facecolor="white",
            alpha=0.9
        )
    )

    # Conteo masculino
    ax.text(
        -row.MALE - max_width*0.03,
        i,
        f"{row.MALE:,}",
        ha="right",
        va="center",
        fontsize=8,
        color=male_color,
        fontweight="bold"
    )

    # Conteo femenino
    ax.text(
        row.FEMALE + max_width*0.03,
        i,
        f"{row.FEMALE:,}",
        ha="left",
        va="center",
        fontsize=8,
        color=female_color,
        fontweight="bold"
    )

# =====================================
# Eje X
# =====================================

ax.xaxis.set_major_formatter(
    mticker.FuncFormatter(
        lambda x, p: f"{abs(int(x)):,}"
    )
)

ax.set_xlabel(
    "Número de pacientes",
    fontsize=11
)

# Grid suave

ax.grid(
    axis="x",
    linestyle="--",
    alpha=0.3
)

# =====================================
# Estética
# =====================================

ax.set_title(
    "Pirámide poblacional de pacientes",
    fontsize=16,
    fontweight="bold"
)

ax.set_ylabel("Grupo etario")

for spine in [
    "top",
    "right"
]:
    ax.spines[spine].set_visible(False)

# Leyenda inferior

ax.legend(
    loc="upper center",
    bbox_to_anchor=(0.5, -0.08),
    ncol=2,
    frameon=False
)

plt.tight_layout()

plt.show()

In [ ]:
for col in [
    "gender",
    "race",
    "ethnicity"
]:

    pct = (
        person[col]
        .isna()
        .mean()
        * 100
    )

    print(
        f"{col}: {pct:.4f}% sin mapeo"
    )

## 1.2 condition_occurrence.csv (sample = 1M)



### **1.2.1 Calidad de condition_ocurrence.csv**

Antes de empezar, debido al tamaño del archivo, se eligio una muestra aleatoria significativa para el analisis. El archivo condition_ocurrence.csv, se define de la siguiente manera:

- condition_occurrence_id: Identificador único de cada diagnóstico o condición clínica registrada.
- person_id: Identificador del paciente asociado al diagnóstico.
- visit_occurrence_id: Identificador de la visita relacionada con el diagnóstico registrado.
- condition_concept_id: Identificador estandarizado del diagnóstico o condición clínica.
- condition_start_datetime: Fecha y hora de inicio o registro de la condición clínica.

In [ ]:
condition_ocurrence = cargar_parquet(
    "condition_occurrence_sample_1M.parquet"
)

In [ ]:
analizar_tabla(condition_ocurrence, "condition_occurrence")

In [ ]:
heatmap_nulos(condition_ocurrence)

In [ ]:
histogramas(condition_ocurrence)

In [ ]:
boxplots(condition_ocurrence)

In [ ]:
correlaciones(condition_ocurrence)

### **1.2.2 Tratamiento de la variable condition_type_concept_id y visit_concurrence_id**

In [ ]:
condition_ocurrence["condition_type_concept_id"] \
    .value_counts(dropna=False)


condition_occurrence_clean = (
    condition_ocurrence
    .drop(columns=["condition_type_concept_id"])
)

La variable presenta un único valor para todos los registros analizados (0).

Al no aportar variabilidad ni capacidad discriminativa, se excluye de los análisis posteriores.

In [ ]:
condition_ocurrence["visit_occurrence_id"].notna().mean() * 100

La variable visit_occurrence_id presentó 20.355 registros nulos (2.04%).

Esto implica que el 97.96% de los diagnósticos mantienen una relación válida con una visita hospitalaria dentro del modelo OMOP.

Aunque existe una pequeña pérdida de integridad referencial, el nivel de completitud observado es alto y permite continuar con el análisis.

### **1.2.3 Exploracion y analisis de los datos de condition_ocurrence.csv**

In [ ]:
[col for col in condition_ocurrence.columns if "concept_id" in col]

In [ ]:
condition_ocurrence["condition"] = (
    condition_ocurrence["condition_concept_id"]
    .map(concept_dict)
)

condition_ocurrence["condition_type"] = (
    condition_ocurrence["condition_type_concept_id"]
    .map(concept_dict)
)

In [ ]:
condition_ocurrence[
    [
        "condition_concept_id",
        "condition"
    ]
].head()

In [ ]:
(
    condition_ocurrence["condition"]
    .value_counts()
    .head(20)
)

In [ ]:
top_conditions = (
    condition_ocurrence["condition"]
    .value_counts()
    .head(15)
)

plt.figure(figsize=(12,6))

sns.barplot(
    x=top_conditions.values,
    y=top_conditions.index
)

plt.title(
    "Top 15 diagnósticos más frecuentes"
)

plt.xlabel("Número de registros")
plt.ylabel("Diagnóstico")

plt.show()

In [ ]:
missing_ids = (
    condition_ocurrence.loc[
        condition_ocurrence["condition"] == "No matching concept",
        "condition_concept_id"
    ]
    .value_counts()
)

missing_ids.head(20)

In [ ]:
diagnosticos_validos = (
    condition_ocurrence[
        condition_ocurrence["condition"]
        != "No matching concept"
    ]
)

top_dx = (
    diagnosticos_validos["condition"]
    .value_counts()
    .head(15)
)

In [ ]:
top_dx = (
    condition_ocurrence
    .query("condition != 'No matching concept'")
    ["condition"]
    .value_counts()
    .head(20)
)

fig, ax = plt.subplots(figsize=(12,6))

sns.barplot(
    x=top_dx.values,
    y=top_dx.index,
    color="#4472C4",
    ax=ax
)

for i, v in enumerate(top_dx.values):
    ax.text(
        v + top_dx.max()*0.01,
        i,
        f"{v:,}",
        va="center",
        fontsize=7
    )

ax.set_title(
    "Top 15 diagnósticos estandarizados más frecuentes",
    fontsize=14,
    fontweight="bold"
)

ax.set_xlabel("Número de registros")
ax.set_ylabel("Diagnóstico")

ax.grid(
    axis="x",
    linestyle="--",
    alpha=0.3
)

plt.tight_layout()

plt.show()

In [ ]:
condition_ocurrence['mes'] = condition_ocurrence['condition_start_datetime'].dt.to_period('M')
condition_ocurrence['dia_semana'] = condition_ocurrence['condition_start_datetime'].dt.day_name()
condition_ocurrence['hora'] = condition_ocurrence['condition_start_datetime'].dt.hour


In [ ]:
meses = (
    condition_ocurrence
    .groupby(condition_ocurrence["condition_start_datetime"].dt.to_period("M"))
    .size()
    .reset_index(name="cantidad")
)

meses["condition_start_datetime"] = meses[
    "condition_start_datetime"
].astype(str)

plt.figure(figsize=(12,5))
sns.lineplot(
    data=meses,
    x="condition_start_datetime",
    y="cantidad",
    marker="o"
)

plt.xticks(rotation=45)
plt.title("Cantidad de diagnósticos por mes")
plt.xlabel("Mes")
plt.ylabel("Número de diagnósticos")
plt.tight_layout()
plt.show()

In [ ]:
orden = [
    "Monday","Tuesday","Wednesday",
    "Thursday","Friday","Saturday","Sunday"
]

dias = (
    condition_ocurrence["condition_start_datetime"]
    .dt.day_name()
    .value_counts()
    .reindex(orden)
)

plt.figure(figsize=(10,5))
sns.barplot(
    x=dias.index,
    y=dias.values,
    palette="Blues_r"
)

plt.title("Diagnósticos por día de la semana")
plt.xlabel("Día")
plt.ylabel("Cantidad")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()


In [ ]:
hora = (
    condition_ocurrence["condition_start_datetime"]
    .dt.hour
    .value_counts()
    .sort_index()
)

plt.figure(figsize=(12,5))
sns.lineplot(
    x=hora.index,
    y=hora.values,
    marker="o"
)

plt.title("Diagnósticos por hora del día")
plt.xlabel("Hora")
plt.ylabel("Cantidad")
plt.xticks(range(24))
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 1.3 procedure_ocurrence.csv (sample = 1M)


### **1.3.1 Calidad de procedure_ocurrence.csv**

- procedure_occurrence_id: Identificador único de cada procedimiento realizado.
- person_id: Identificador del paciente asociado al procedimiento.
- visit_occurrence_id: Identificador de la visita asociada al procedimiento realizado.
- procedure_concept_id: Identificador estandarizado del procedimiento clínico realizado.
- procedure_datetime: Fecha y hora de realización del procedimiento.
- procedure_type_concept_id: Identificador asociado al tipo u origen del procedimiento registrado.

In [ ]:
procedure_ocurrence = cargar_parquet(
    "procedure_occurrence_sample_1M.parquet"
)

In [ ]:
analizar_tabla(procedure_ocurrence, "procedure_occurrence")

In [ ]:
heatmap_nulos(procedure_ocurrence)

In [ ]:
histogramas(procedure_ocurrence)

In [ ]:
boxplots(procedure_ocurrence)

In [ ]:
correlaciones(procedure_ocurrence)

### **1.3.2 Tratamiento de la variable procedure_date y visit_occurrence_id**

In [ ]:
procedure_ocurrence["procedure_date"].isna().mean() * 100

La variable procedure_date presenta un 100% de valores nulos, por lo que no fue considerada en el análisis temporal de procedimientos.

In [ ]:
procedure_ocurrence = procedure_ocurrence.drop(columns=["procedure_date"])

In [ ]:
procedure_ocurrence["visit_occurrence_id"].isna().mean() * 100

In [ ]:
procedure_ocurrence[procedure_ocurrence["visit_occurrence_id"].isna()].head()

visit_occurrence_id:
- 0.084% de valores nulos.
- Se conserva la variable debido a su alta completitud.
- Los registros sin identificador de visita se excluyen únicamente en análisis que requieren vinculación con la tabla visit_occurrence.

### **1.3.3 Exploracion y analisis de datos de procedure_ocurrence.csv**

In [ ]:
concept = cargar_csv(
    "concept.csv"
)

concept_dict = dict(
    zip(
        concept["concept_id"],
        concept["concept_name"]
    )
)

In [ ]:
[col for col in procedure_ocurrence.columns if "concept_id" in col]

In [ ]:
procedure_ocurrence["procedure"] = (
    procedure_ocurrence["procedure_concept_id"]
    .map(concept_dict)
)

In [ ]:
(
    procedure_ocurrence["procedure"]
    .value_counts()
    .head(20)
)

In [ ]:
top_dx = (
    procedure_ocurrence
    .query("procedure != 'No matching concept'")
    ["procedure"]
    .value_counts()
    .head(20)
)

top_dx.index = [
    texto[:40] + "..."
    if len(texto) > 40
    else texto
    for texto in top_dx.index
]

fig, ax = plt.subplots(figsize=(12,6))

sns.barplot(
    x=top_dx.values,
    y=top_dx.index,
    color="#4472C4",
    ax=ax
)

for i, v in enumerate(top_dx.values):
    ax.text(
        v + top_dx.max()*0.01,
        i,
        f"{v:,}",
        va="center",
        fontsize=7
    )

ax.set_title(
    "Top 15 procedimientos estandarizados más frecuentes",
    fontsize=14,
    fontweight="bold"
)

ax.set_xlabel("Número de registros")
ax.set_ylabel("Procedimiento")

ax.grid(
    axis="x",
    linestyle="--",
    alpha=0.3
)

plt.tight_layout()

plt.show()

In [ ]:
(
    procedure_ocurrence
    .groupby("visit_occurrence_id")
    .size()
)

## 1.4 measurement (sample = 1M)
 


- measurement_id: Identificador único de cada medición clínica registrada.
- person_id: Identificador del paciente asociado a la medición clínica.
- visit_occurrence_id: Identificador de la visita asociada a la medición realizada.
- measurement_concept_id: Identificador estandarizado de la medición clínica o laboratorio realizado.
- measurement_datetime: Fecha y hora de realización de la medición clínica.
- value_as_number: Valor numérico asociado al resultado de la medición clínica.
- unit_concept_id: Identificador estandarizado de la unidad de medida utilizada.
- measurement_type_concept_id: Identificador asociado al tipo u origen de la medición registrada.

In [ ]:
measurement = cargar_parquet(
    "measurement_sample_1M.parquet"
)

In [ ]:
analizar_tabla(measurement, "measurement")

In [ ]:
heatmap_nulos(measurement)

In [ ]:
histogramas(measurement)

In [ ]:
boxplots(measurement)

In [ ]:
correlaciones(measurement)

## 1.5 visit_occurrence.csv



- visit_occurrence_id: Identificador único de cada visita o encuentro clínico del paciente.
- person_id: Identificador del paciente asociado a la visita.
- visit_concept_id: Identificador estandarizado del tipo de visita (urgencias, hospitalización, consulta, etc.).
- visit_start_datetime: Fecha y hora de inicio de la visita hospitalaria.
- visit_end_datetime: Fecha y hora de finalización de la visita hospitalaria.
- visit_type_concept_id: Identificador que describe el origen o tipo de registro de la visita.
- provider_id: Identificador del profesional de salud asociado a la visita.

In [ ]:
visit_ocurrence = cargar_csv(
    "visit_occurrence.csv"
)

In [ ]:
analizar_tabla(visit_ocurrence, "visit_occurrence")

In [ ]:
heatmap_nulos(visit_ocurrence)

In [ ]:
histogramas(visit_ocurrence)

In [ ]:
boxplots(visit_ocurrence)

In [ ]:
correlaciones(visit_ocurrence)

## 1.6 visit_detail.csv 


- visit_detail_id: Identificador único de cada detalle o microproceso dentro de una visita hospitalaria.
- visit_occurrence_id: Identificador de la visita principal asociada al detalle registrado.
- visit_detail_concept_id: Identificador estandarizado del tipo de etapa o subproceso clínico dentro de la visita.
- visit_detail_start_datetime: Fecha y hora de inicio del subproceso o etapa hospitalaria.
- visit_detail_end_datetime: Fecha y hora de finalización del subproceso o etapa hospitalaria.
- provider_id: Identificador del profesional asociado al subproceso registrado.

In [ ]:
visit_detail = cargar_csv(
    "visit_detail.csv"
)

In [ ]:
analizar_tabla(visit_detail, "visit_detail")

In [ ]:
heatmap_nulos(visit_detail)

In [ ]:
histogramas(visit_detail)

In [ ]:
boxplots(visit_detail)

In [ ]:
correlaciones(visit_detail)

# **2. Modelo Relacional de los Datos**

# **3. Ingenieria de Variables**